# Predicting the Source of a News Headline — with proper evaluation

Same idea as `MAIN FILE.ipynb` (Doc2Vec on headlines from 6 publishers), but now with:
1. A **train/test split** (the test headlines are never seen during training)
2. **Method 1** — the original approach (one vector per publisher + cosine similarity), now measured properly
3. **Method 2** — Doc2Vec vector for every headline + **Logistic Regression** and an **ANN (neural network)**
4. A **TF-IDF** baseline for comparison
5. Accuracy, precision, recall and F1 for every method

In [1]:
import re, time
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

DATA_PATH = "uci-news-aggregator.csv"  # download from Kaggle (link in README)
SEED = 42

## 1. Load data and keep the 6 publishers with 2,000–3,000 headlines (same filter as before)

In [2]:
df = pd.read_csv(DATA_PATH)
news = df[["TITLE", "PUBLISHER"]].dropna()

counts = news["PUBLISHER"].value_counts()
publishers = counts[(counts > 2000) & (counts < 3000)].index
news = news[news["PUBLISHER"].isin(publishers)].reset_index(drop=True)

print("Total headlines in dataset:", len(df))
print("Headlines kept:", len(news))
print(news["PUBLISHER"].value_counts())

Total headlines in dataset: 422419
Headlines kept: 13751
PUBLISHER
Huffington Post     2455
Businessweek        2395
Contactmusic.com    2334
Daily Mail          2254
NASDAQ              2228
Examiner.com        2085
Name: count, dtype: int64


## 2. Clean text: lowercase, remove punctuation, split into words

In [3]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9$%' ]", " ", text)   # keep letters, numbers, $ % '
    return text.split()

news["tokens"] = news["TITLE"].apply(clean)
news[["TITLE", "tokens"]].head()

,TITLE,tokens
0,Plosser: Fed May Have to Accelerate Tapering Pace,"[plosser, fed, may, have, to, accelerate, tape..."
1,Forex - Pound drops to one-month lows against ...,"[forex, pound, drops, to, one, month, lows, ag..."
2,"Euro Anxieties Wane as Bunds Top Treasuries, S...","[euro, anxieties, wane, as, bunds, top, treasu..."
3,Noyer Says Strong Euro Creates Unwarranted Eco...,"[noyer, says, strong, euro, creates, unwarrant..."
4,EBay CEO Donahoe's Pay Drops by 53% to $13.8 M...,"[ebay, ceo, donahoe's, pay, drops, by, 53%, to..."


## 3. Train / test split (80 / 20, stratified so every publisher has the same share in both)

In [4]:
train, test = train_test_split(news, test_size=0.2, stratify=news["PUBLISHER"], random_state=SEED)
y_train, y_test = train["PUBLISHER"].values, test["PUBLISHER"].values
print("Train:", len(train), " Test:", len(test))

results = []
def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    results.append({"Method": name, "Accuracy": acc, "Precision (macro)": p, "Recall (macro)": r, "F1 (macro)": f1})
    print(f"{name}: accuracy={acc:.3f}  precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}")

Train: 11000  Test: 2751


## 4. Method 1 — original approach: one vector per publisher + cosine similarity

Each headline is tagged with its **publisher name**, so Doc2Vec learns 6 publisher vectors.
A test headline is turned into a vector (`infer_vector`) and gets the publisher whose vector is most similar.

In [5]:
t = time.time()
pub_docs = [TaggedDocument(tok, [pub]) for tok, pub in zip(train["tokens"], train["PUBLISHER"])]
pub_model = Doc2Vec(dm=0, vector_size=100, min_count=2, epochs=30, workers=4, seed=SEED)
pub_model.build_vocab(pub_docs)
pub_model.train(pub_docs, total_examples=pub_model.corpus_count, epochs=pub_model.epochs)

pred_sim = [pub_model.dv.most_similar([pub_model.infer_vector(tok)], topn=1)[0][0] for tok in test["tokens"]]
evaluate("Doc2Vec + cosine similarity (original)", y_test, pred_sim)
print(f"time {time.time()-t:.0f}s")

Doc2Vec + cosine similarity (original): accuracy=0.526  precision=0.543  recall=0.527  F1=0.497
time 21s


## 5. Method 2 — one vector per headline, then a classifier

Each headline gets its own tag, so Doc2Vec (PV-DBOW) learns a 100-number vector for every headline.
Train and test headlines are both converted with `infer_vector`, so they are treated the same way.

In [6]:
t = time.time()
doc_docs = [TaggedDocument(tok, [i]) for i, tok in enumerate(train["tokens"])]
d2v = Doc2Vec(dm=0, vector_size=100, min_count=2, epochs=30, workers=4, seed=SEED)
d2v.build_vocab(doc_docs)
d2v.train(doc_docs, total_examples=d2v.corpus_count, epochs=d2v.epochs)

X_train = np.array([d2v.infer_vector(tok) for tok in train["tokens"]])
X_test = np.array([d2v.infer_vector(tok) for tok in test["tokens"]])

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
print("Vector shape:", X_train.shape, f" time {time.time()-t:.0f}s")

Vector shape: (11000, 100)  time 23s


In [7]:
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train_s, y_train)
evaluate("Doc2Vec + Logistic Regression", y_test, lr.predict(X_test_s))

Doc2Vec + Logistic Regression: accuracy=0.501  precision=0.502  recall=0.499  F1=0.498


### ANN (neural network)
Input: 100 Doc2Vec numbers → hidden layer of 128 neurons → hidden layer of 64 neurons → output layer of 6 (one per publisher, softmax).
ReLU activation, Adam optimizer, early stopping (stops when the score on a 10% validation slice stops improving).

In [8]:
ann = MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
                    alpha=1e-3, early_stopping=True, validation_fraction=0.1,
                    max_iter=300, random_state=SEED)
# the ANN needs number labels: 0..5 instead of publisher names
enc = LabelEncoder().fit(y_train)
ann.fit(X_train_s, enc.transform(y_train))
evaluate("Doc2Vec + ANN", y_test, enc.inverse_transform(ann.predict(X_test_s)))
print("Epochs trained:", ann.n_iter_)

Doc2Vec + ANN: accuracy=0.517  precision=0.527  recall=0.514  F1=0.514
Epochs trained: 20


## 6. Baseline for comparison — TF-IDF + Logistic Regression

In [9]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
T_train = tfidf.fit_transform(train["tokens"].str.join(" "))
T_test = tfidf.transform(test["tokens"].str.join(" "))
tf_lr = LogisticRegression(max_iter=2000, C=5)
tf_lr.fit(T_train, y_train)
evaluate("TF-IDF + Logistic Regression", y_test, tf_lr.predict(T_test))

TF-IDF + Logistic Regression: accuracy=0.629  precision=0.630  recall=0.629  F1=0.629


## 7. Results (all on the unseen test set)

In [10]:
res = pd.DataFrame(results).set_index("Method").round(3)
res

,Accuracy,Precision (macro),Recall (macro),F1 (macro)
Method,,,,
Doc2Vec + cosine similarity (original),0.526,0.543,0.527,0.497
Doc2Vec + Logistic Regression,0.501,0.502,0.499,0.498
Doc2Vec + ANN,0.517,0.527,0.514,0.514
TF-IDF + Logistic Regression,0.629,0.630,0.629,0.629


## 8. Detailed report for the ANN

In [11]:
ann_pred = enc.inverse_transform(ann.predict(X_test_s))
print(classification_report(y_test, ann_pred, digits=3))
labels = sorted(news["PUBLISHER"].unique())
pd.DataFrame(confusion_matrix(y_test, ann_pred, labels=labels), index=labels, columns=labels)

                  precision    recall  f1-score   support

    Businessweek      0.630     0.633     0.631       479
Contactmusic.com      0.489     0.473     0.481       467
      Daily Mail      0.507     0.424     0.461       451
    Examiner.com      0.465     0.288     0.356       417
 Huffington Post      0.339     0.487     0.400       491
          NASDAQ      0.730     0.783     0.755       446

        accuracy                          0.517      2751
       macro avg      0.527     0.514     0.514      2751
    weighted avg      0.525     0.517     0.515      2751



,Businessweek,Contactmusic.com,Daily Mail,Examiner.com,Huffington Post,NASDAQ
Businessweek,303,9,16,4,79,68
Contactmusic.com,13,221,60,45,115,13
Daily Mail,38,75,191,32,105,10
Examiner.com,32,62,50,120,137,16
Huffington Post,42,80,56,52,239,22
NASDAQ,53,5,4,5,30,349


## 9. Try a new headline

In [12]:
def predict_source(headline):
    vec = scaler.transform([d2v.infer_vector(clean(headline))])
    probs = ann.predict_proba(vec)[0]
    top = np.argsort(probs)[::-1][:3]
    return [(enc.classes_[i], round(float(probs[i]), 3)) for i in top]

predict_source("Stocks fall as Fed hints at faster tapering")

[('NASDAQ', 0.579), ('Businessweek', 0.419), ('Huffington Post', 0.001)]